In [9]:
import shutil
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torchvision
import kagglehub

In [10]:
DATA_ROOT = Path('/Users/ilya/Documents/CV_train_DL/data')
BATCHES_DIR = DATA_ROOT / 'cifar-10-batches-py'


def ensure_cifar10_from_kaggle(data_root: Path = DATA_ROOT) -> Path:
    """Download CIFAR-10 from Kaggle and place batches under data_root."""
    data_root.mkdir(parents=True, exist_ok=True)
    batches_dir = data_root / 'cifar-10-batches-py'

    if (batches_dir / 'data_batch_1').exists() and (batches_dir / 'test_batch').exists():
        return batches_dir

    # Drop incomplete torchvision download if present.
    broken_archive = data_root / 'cifar-10-python.tar.gz'
    if broken_archive.exists() and broken_archive.stat().st_size < 170_000_000:
        broken_archive.unlink()

    cache_path = Path(
        kagglehub.dataset_download(
            'pankrzysiu/cifar10-python',
            output_dir=str(data_root / 'kaggle_cifar10'),
        )
    )

    # Dataset may contain extracted batches and/or the .tar.gz archive.
    candidates = [
        cache_path / 'cifar-10-batches-py',
        cache_path / 'cifar10-python' / 'cifar-10-batches-py',
    ]
    for candidate in candidates:
        if (candidate / 'data_batch_1').exists():
            if batches_dir.exists():
                shutil.rmtree(batches_dir)
            shutil.copytree(candidate, batches_dir)
            return batches_dir

    archives = list(cache_path.rglob('cifar-10-python.tar.gz'))
    if not archives:
        raise FileNotFoundError(f'CIFAR-10 archive not found under {cache_path}')

    with tarfile.open(archives[0], 'r:gz') as tar:
        tar.extractall(path=data_root)

    if not (batches_dir / 'data_batch_1').exists():
        raise FileNotFoundError(f'Failed to extract CIFAR-10 batches into {batches_dir}')
    return batches_dir


ensure_cifar10_from_kaggle()

train_transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomCrop(32, padding=4),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

train_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=True,
                             download=False, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=False,
                            download=False, transform=test_transform)

print(len(train_set), len(test_set), train_set.classes)



50000 10000 ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [11]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),)
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),)
        self.block3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),)
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(2048, 512)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(512, 10)
        
    def forward(self, X):
        X = self.block1(X)
        X = self.block2(X)
        X = self.block3(X)
        X = self.flatten(X)
        X = self.linear1(X)
        X = self.relu(X)
        X = self.linear2(X)
        return X


In [ ]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)
sheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=100)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

for epoch in range(100):
    model.train()
    train_loss=0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss = train_loss/len(train_set)
    sheduler.step()
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            output = model(x)
            loss = criterion(output, y)
            val_loss += loss.item() * x.size(0)
            correct += (output.argmax(1) == y).sum().item()
            total += y.size(0)
    
    val_loss /= len(test_set)
    val_acc = correct / total
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

Epoch 0: train_loss=1.6043, val_loss=1.3992, val_acc=0.4739
Epoch 1: train_loss=1.2598, val_loss=1.2743, val_acc=0.5559
Epoch 2: train_loss=1.1107, val_loss=1.2120, val_acc=0.5731
Epoch 3: train_loss=1.0195, val_loss=1.7366, val_acc=0.4831
Epoch 4: train_loss=0.9524, val_loss=1.1126, val_acc=0.6158
Epoch 5: train_loss=0.9012, val_loss=0.9602, val_acc=0.6646
Epoch 6: train_loss=0.8580, val_loss=0.9302, val_acc=0.6808
Epoch 7: train_loss=0.8179, val_loss=0.8207, val_acc=0.7068
Epoch 8: train_loss=0.7868, val_loss=1.1128, val_acc=0.6285
Epoch 9: train_loss=0.7567, val_loss=0.7374, val_acc=0.7384
Epoch 10: train_loss=0.7284, val_loss=0.7627, val_acc=0.7376
Epoch 11: train_loss=0.7103, val_loss=0.7396, val_acc=0.7422
Epoch 12: train_loss=0.6834, val_loss=0.9804, val_acc=0.6786
Epoch 13: train_loss=0.6717, val_loss=0.7304, val_acc=0.7479
Epoch 14: train_loss=0.6503, val_loss=0.6321, val_acc=0.7753
Epoch 15: train_loss=0.6301, val_loss=0.8165, val_acc=0.7253
Epoch 16: train_loss=0.6173, val_l

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [13]:
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)



train_transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomCrop(32, padding=4),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=imagenet_mean,
                         std=imagenet_std),
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=imagenet_mean,
                         std=imagenet_std),
])

train_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=True,
                             download=False, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=False,
                            download=False, transform=test_transform)



model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

for param in model.parameters():
    param.requires_grad = False
for param in model.conv1.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True
for param in model.bn1.parameters():
    param.requires_grad = True
optimizer = optim.SGD([
    {'params': model.layer4.parameters(), 'lr': 1e-4, 'weight_decay':1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3, 'weight_decay':0},
    {'params': model.conv1.parameters(), 'lr': 1e-3, 'weight_decay':1e-4},
    {'params': model.bn1.parameters(), 'lr': 1e-3, 'weight_decay':0},
], momentum=0.9)

sheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

for epoch in range(30):
    model.train()
    train_loss=0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss = train_loss/len(train_set)
    sheduler.step()
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output, y)
            val_loss += loss.item() * x.size(0)
            correct += (output.argmax(1) == y).sum().item()
            total += y.size(0)
    
    val_loss /= len(test_set)
    val_acc = correct / total
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

Epoch 0: train_loss=1.5516, val_loss=1.2545, val_acc=0.5610
Epoch 1: train_loss=1.1096, val_loss=0.9793, val_acc=0.6574
Epoch 2: train_loss=0.9584, val_loss=0.8626, val_acc=0.7022
Epoch 3: train_loss=0.8641, val_loss=0.8065, val_acc=0.7185
Epoch 4: train_loss=0.8013, val_loss=0.7444, val_acc=0.7411
Epoch 5: train_loss=0.7553, val_loss=0.7052, val_acc=0.7541
Epoch 6: train_loss=0.7188, val_loss=0.6805, val_acc=0.7620
Epoch 7: train_loss=0.6828, val_loss=0.6410, val_acc=0.7777
Epoch 8: train_loss=0.6535, val_loss=0.6253, val_acc=0.7792
Epoch 9: train_loss=0.6290, val_loss=0.6040, val_acc=0.7869
Epoch 10: train_loss=0.6155, val_loss=0.5828, val_acc=0.7936
Epoch 11: train_loss=0.5903, val_loss=0.5550, val_acc=0.8041
Epoch 12: train_loss=0.5748, val_loss=0.5447, val_acc=0.8109
Epoch 13: train_loss=0.5610, val_loss=0.5427, val_acc=0.8104
Epoch 14: train_loss=0.5452, val_loss=0.5135, val_acc=0.8247
Epoch 15: train_loss=0.5339, val_loss=0.5102, val_acc=0.8258
Epoch 16: train_loss=0.5233, val_l